# 04a · Aggregate 2025 Cooling Assistance applications to modified ZIP code areas

**Objective.** Turn the NYC Department of Social Services (DSS) workbook of Cooling Assistance approvals and
denials by ZIP code into 2025 application counts per modified ZIP code area (MODZCTA), the geography used in
`analysis.ipynb`.

The workbook was obtained by The Margin and is **not redistributed** with this repository. This notebook is
kept so the aggregation is inspectable and rerunnable by anyone who has the file. Its three outputs are
redistributed: `../inputs/dss_applications_2025_by_modzcta.csv`,
`../inputs/dss_applications_2025_unmatched_zips.csv` and `../inputs/dss_applications_2025_totals.json`. Approved and denied counts are summed into application totals
here and are not used separately anywhere in the analysis; denial reasons are not used at all.

Set the workbook path in the next cell (or the `DSS_WORKBOOK` environment variable).

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from its own directory (run_notebooks.py does)
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)

import os

import pandas as pd

import json

from common import INPUTS

WORKBOOK = Path(os.environ.get("DSS_WORKBOOK", ROOT.parent / "Cooling Assistance Application Approvals and Denials by Borough (2024, 2025).xlsx"))
if not WORKBOOK.exists():
    raise FileNotFoundError(f"DSS workbook not found at {WORKBOOK}. Set DSS_WORKBOOK to its path; the file is not part of this repository.")

MANUAL_ZIP_TO_MODZCTA = {"11249": "11211"}  # Source: NYC Open Data pri4-ifjk, feature 11211, label "11211, 11249"


## 1. Read the raw pivot sheets

The sheets `Approved (2025)` and `Denied (2025)` are pivot tables: one row per ZIP code, grouped under borough
subtotal rows (the five boroughs plus an OTHER bucket for non-NYC ZIPs), with a grand total. Rows whose label is a
five-digit ZIP code are kept and summed per ZIP; the workbook records borough independently of ZIP, so the same
ZIP can appear under more than one borough. Every other row is classified: subtotal rows are dropped, and any
remaining row that carries a count is a record with no usable ZIP label. The kept rows plus those unlabeled
counts must equal the sheet's grand total, so nothing can disappear unnoticed. In the 2025 Denied sheet one row
labeled `NULL` carries 1 denial; it is reported and excluded from the ZIP-level table because it cannot be placed.

In [2]:
SUBTOTAL_LABELS = {"bronx", "brooklyn", "manhattan", "queens", "staten island", "other"}


def read_sheet(sheet: str, value_column: str, name: str):
    """Return (per-ZIP sums, grand total, count on rows with no usable ZIP label) for one pivot sheet."""
    frame = pd.read_excel(WORKBOOK, sheet_name=sheet)
    label = frame.iloc[:, 0].astype("string").str.strip().fillna("")   # a NULL/blank label reads as missing
    values = pd.to_numeric(frame[value_column], errors="coerce").fillna(0)
    is_zip = label.str.fullmatch(r"\d{5}").fillna(False)
    is_total = label.str.lower() == "grand total"
    is_subtotal = label.str.lower().isin(SUBTOTAL_LABELS)
    unlabeled = values[~is_zip & ~is_total & ~is_subtotal]
    grand_total = int(values[is_total].iloc[0])
    by_zip = pd.DataFrame({"zip": label[is_zip], name: values[is_zip].astype(int)}).groupby("zip", as_index=False)[name].sum()
    assert int(by_zip[name].sum()) + int(unlabeled.sum()) == grand_total, (sheet, int(by_zip[name].sum()), int(unlabeled.sum()), grand_total)
    if unlabeled.sum():
        print(f"{sheet}: {int(unlabeled.sum())} record(s) on {int((unlabeled > 0).sum())} row(s) with no usable ZIP label "
              f"(labels: {sorted(set(label[unlabeled.index][unlabeled > 0]))}); excluded from the ZIP table")
    return by_zip, grand_total, int(unlabeled.sum())


approved, approved_total, approved_unlabeled = read_sheet("Approved (2025)", "Count of Zip", "approved")
denied, denied_total, denied_unlabeled = read_sheet("Denied (2025)", "Grand Total", "denied")
by_zip = approved.merge(denied, on="zip", how="outer")          # a ZIP may appear on only one sheet
by_zip[["approved", "denied"]] = by_zip[["approved", "denied"]].fillna(0).astype(int)
by_zip["applications_2025"] = by_zip["approved"] + by_zip["denied"]
workbook_total = approved_total + denied_total
without_zip = approved_unlabeled + denied_unlabeled
total = int(by_zip.applications_2025.sum())
assert total + without_zip == workbook_total
print(f"workbook grand totals: {workbook_total:,} applications; {total:,} on {len(by_zip)} five-digit ZIP rows; {without_zip} with no usable ZIP label")


Denied (2025): 1 record(s) on 1 row(s) with no usable ZIP label (labels: ['']); excluded from the ZIP table
workbook grand totals: 26,622 applications; 26,621 on 190 five-digit ZIP rows; 1 with no usable ZIP label


## 2. Map ZIP codes to modified ZIP code areas

The DOHMH ZCTA-to-MODZCTA crosswalk assigns each ZIP to one MODZCTA. One ZIP, 11249 (North Williamsburg), is
absent from the crosswalk file; the MODZCTA dataset itself labels feature 11211 as "11211, 11249", so that
assignment is added. Counts only ever aggregate upward: no ZIP total is split between areas. ZIP labels with no MODZCTA in the
crosswalk are written out rather than dropped. Most are identifiable as ZIPs outside New York City or PO-box
ZIPs; a few labels do not correspond to any assigned ZIP code and may be mistyped. The totals file records the
full reconciliation from the workbook's grand totals to the matched count.

In [3]:
crosswalk = pd.read_csv(INPUTS / "zcta_to_modzcta.csv", dtype=str)
lookup = dict(zip(crosswalk["ZCTA"], crosswalk["MODZCTA"])) | MANUAL_ZIP_TO_MODZCTA
by_zip["modzcta"] = by_zip["zip"].map(lookup)
matched = by_zip[by_zip.modzcta.notna()]
unmatched = by_zip[by_zip.modzcta.isna()][["zip", "applications_2025"]].sort_values("applications_2025", ascending=False)
assert matched.applications_2025.sum() + unmatched.applications_2025.sum() == total   # nothing lost in the split

by_modzcta = matched.groupby("modzcta", as_index=False)["applications_2025"].sum().sort_values("modzcta")
by_modzcta.to_csv(INPUTS / "dss_applications_2025_by_modzcta.csv", index=False)
unmatched.to_csv(INPUTS / "dss_applications_2025_unmatched_zips.csv", index=False)
totals = {
    "description": "Reconciliation of 2025 Cooling Assistance applications (approved plus denied) from the DSS workbook's sheet grand totals to the counts mapped to modified ZIP code areas.",
    "workbook_grand_total": workbook_total,
    "rows_without_usable_zip_label": without_zip,
    "with_five_digit_zip": total,
    "matched_to_modzcta": int(by_modzcta.applications_2025.sum()),
    "modzcta_areas_with_applications": len(by_modzcta),
    "unmatched_zip_labels": len(unmatched),
    "unmatched_applications": int(unmatched.applications_2025.sum()),
}
(INPUTS / "dss_applications_2025_totals.json").write_text(json.dumps(totals, indent=2) + "\n", encoding="utf-8")
print(f"{int(by_modzcta.applications_2025.sum()):,} applications matched to {len(by_modzcta)} MODZCTAs; "
      f"{int(unmatched.applications_2025.sum()):,} applications in {len(unmatched)} unmatched ZIP codes")
unmatched


26,606 applications matched to 173 MODZCTAs; 15 applications in 13 unmatched ZIP codes


,zip,applications_2025
82,10705,2
88,11096,2
78,10545,1
80,10701,1
79,10550,1
81,10703,1
83,10927,1
97,11202,1
135,11306,1
182,11453,1
